# CKD — UCI Benchmark

400-record public UCI CKD dataset: preprocessing, feature selection, model benchmarking, and stacking.

**Research article:** Ahmed, Rakib, Shiddik & Reza (2025), *International Journal of Statistical Sciences*, 25(2), 1–32. DOI: [10.3329/ijss.v25i2.85732](https://doi.org/10.3329/ijss.v25i2.85732).

**Scientific provenance:** Converted from the user-provided Jupyter HTML export. Core numerical modelling logic and default 2,000 bootstrap resamples are retained; local paths and Seaborn compatibility have been adjusted. See `CONVERSION_NOTES.md` and `METHODS_AND_LIMITATIONS.md`.

**Run instructions:** Start Jupyter in the **repository root**. Execute cells from top to bottom. The original UCI study requires `python scripts/fetch_uci_ckd.py` first; the synthetic study generates its own local data. No notebook outputs are committed. Not a clinical diagnostic tool.


## CKD Paper

## UCI CKD Dataset

In [ ]:
# =========================================================
# CKD: RAW EDA + Early Encoding (No *_yes columns) + Train-only Z-cap
# + SMOTE (train only) + RF Feature Importance
# (pandas-compatible + labeled bars + gradient FI with colorbar)
# =========================================================

# -------- 0) Imports & Config --------
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# for attractive FI colors
import matplotlib.cm as cm
import matplotlib.colors as mcolors

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.base import BaseEstimator, TransformerMixin

from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
from pathlib import Path
import os
CSV_PATH = Path(os.getenv("CKD_UCI_CSV", str(Path("data") / "uci_ckd.csv"))).expanduser()
if not CSV_PATH.is_file():
    raise FileNotFoundError(
        f"UCI CKD CSV not found: {CSV_PATH}. "
        "Run: python scripts/fetch_uci_ckd.py (from repository root), "
        "or set CKD_UCI_CSV to an authorized local CKD CSV file."
    )
plt.rcParams.update({"figure.dpi": 130})


# -------- 1) Helper: Z-score winsorizer (numeric only) --------
class ZScoreWinsorizer(BaseEstimator, TransformerMixin):
    """Train-only z-score capping at [μ- kσ, μ+ kσ] for numeric (continuous) columns."""
    def __init__(self, k=3.0):
        self.k = float(k)
        self.means_ = None
        self.stds_  = None
        self.columns_ = None

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.columns_ = list(X.columns)
        self.means_ = X.mean(axis=0).to_numpy()
        self.stds_  = X.std(ddof=0, axis=0).to_numpy()
        return self

    def transform(self, X):
        was_df = isinstance(X, pd.DataFrame)
        X = pd.DataFrame(X, columns=self.columns_).copy()
        lower = self.means_ - self.k * self.stds_
        upper = self.means_ + self.k * self.stds_
        mask = (self.stds_ == 0)
        if mask.any():
            lower = lower.copy(); upper = upper.copy()
            lower[mask] = -np.inf; upper[mask] =  np.inf
        X = X.clip(lower=lower, upper=upper, axis=1)
        return X if was_df else X.to_numpy()


# -------- 2) Load & literal cleaning (no learned params) --------
df = pd.read_csv(CSV_PATH)
if 'id' in df.columns: df.drop('id', axis=1, inplace=True)

df.columns = [
    'age', 'blood pressure', 'specific gravity', 'albumin', 'sugar', 'red blood cells',
    'pus cell', 'pus cell clumps', 'bacteria', 'blood glucose random', 'blood urea',
    'serum creatinine', 'sodium', 'potassium', 'hemoglobin', 'packed cell volume',
    'white blood cell count', 'red blood cell count', 'hypertension', 'diabetes mellitus',
    'coronary artery disease', 'appetite', 'pedal edema', 'anemia', 'class'
]

# Coerce numeric-like fields to numeric
for c in ['packed cell volume','white blood cell count','red blood cell count',
          'specific gravity','albumin','sugar']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# Fix textual glitches
df['diabetes mellitus']       = df['diabetes mellitus'].replace({'\tno':'no','\tyes':'yes',' yes':'yes'})
df['coronary artery disease'] = df['coronary artery disease'].replace({'\tno':'no'})
df['class']                   = df['class'].replace({'ckd\t':'ckd','notckd':'not ckd'})

# Target mapping (ckd=1)
df['class'] = df['class'].map({'ckd':1, 'not ckd':0})

# Feature groups
NUM_CONTINUOUS = [  # continuous numeric features (z-cap + minmax)
    'age','blood pressure','blood glucose random','blood urea','serum creatinine',
    'sodium','potassium','hemoglobin','packed cell volume',
    'white blood cell count','red blood cell count','specific gravity','albumin','sugar'
]
BIN_CATEGORICAL = [  # binary categoricals to map to 0/1 (no OHE)
    'red blood cells','pus cell','pus cell clumps','bacteria',
    'hypertension','diabetes mellitus','coronary artery disease','appetite','pedal edema','anemia'
]

# Binary mapping dictionaries (1 = “risk”/“presence”)
binary_maps = {
    'hypertension':            {'no':0, 'yes':1},
    'diabetes mellitus':       {'no':0, 'yes':1},
    'coronary artery disease': {'no':0, 'yes':1},
    'pedal edema':             {'no':0, 'yes':1},
    'anemia':                  {'no':0, 'yes':1},
    'appetite':                {'good':0, 'poor':1},
    'red blood cells':         {'normal':0, 'abnormal':1},
    'pus cell':                {'normal':0, 'abnormal':1},
    'pus cell clumps':         {'notpresent':0, 'present':1},
    'bacteria':                {'notpresent':0, 'present':1},
}


# -------- 3) RAW EDA (no learned transforms) --------
print("\n[RAW] Dataset info:")
df.info()

print("\n[RAW] Statistical summary (numeric):")
num_summary = df.select_dtypes(include=[np.number]).describe()
print(num_summary)

print("\n[RAW] Missing values per column:")
missing_series = df.isna().sum().sort_values(ascending=False)
print(missing_series)

# Missing values bar plot with labels on top
plt.figure(figsize=(12,5))
ax = sns.barplot(x=missing_series.index, y=missing_series.values, color='royalblue')
plt.title("Missing Values per Feature (RAW)")
plt.ylabel("Count"); plt.xlabel("Feature")
plt.xticks(rotation=45, ha='right')

# value labels on each bar (use the same ordering as missing_series)
for p, v in zip(ax.patches, missing_series.values):
    ax.annotate(f'{int(v)}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=10, xytext=(0, 3), textcoords='offset points')

plt.tight_layout(); plt.show()

# Numeric distributions (RAW)
plt.figure(figsize=(16, 12))
for i, c in enumerate(NUM_CONTINUOUS, start=1):
    plt.subplot(5, 3, i)
    sns.histplot(df[c], kde=True, bins=30, color='dodgerblue', stat="density", linewidth=1)
    plt.title(c, fontsize=9); plt.tight_layout()
plt.suptitle("Numerical Features — RAW Distributions", fontsize=14, y=1.02); plt.show()

# Categorical distributions (RAW)
plt.figure(figsize=(16, 12))
for i, c in enumerate(BIN_CATEGORICAL, start=1):
    plt.subplot(4, 3, i)
    ax = sns.countplot(x=c, data=df, hue=c, palette="Set2", legend=False)
    plt.title(c, fontsize=9); plt.xticks(rotation=45, fontsize=8)
    for p in ax.patches:
        h = p.get_height()
        ax.annotate(f'{int(h)}', (p.get_x()+p.get_width()/2., h),
                    ha='center', va='bottom', fontsize=7, xytext=(0,3), textcoords='offset points')
plt.tight_layout(rect=[0,0,1,0.98]); plt.suptitle("Binary Categorical — RAW Distributions", fontsize=14, y=0.99)
plt.show()

# Robust numeric correlation (RAW) with numbers — numeric-only
numeric_df = df[NUM_CONTINUOUS].apply(pd.to_numeric, errors='coerce')
usable_cols = [c for c in numeric_df.columns if numeric_df[c].notna().sum() >= 2]
if len(usable_cols) >= 2:
    corr = numeric_df[usable_cols].corr(min_periods=2).round(2)
    mask = corr.isna()
    plt.figure(figsize=(max(8, 0.8*len(usable_cols)), max(6, 0.7*len(usable_cols))))
    sns.heatmap(corr, mask=mask, cmap='coolwarm', vmin=-1, vmax=1, center=0,
                annot=True, fmt=".2f", annot_kws={"size":8}, linewidths=.3, cbar_kws={'shrink':0.8})
    plt.title("Correlation Heatmap (Numerical Features)"); plt.tight_layout(); plt.show()
else:
    print("[INFO] Not enough usable numeric columns for correlation heatmap.")

# Correlation with target (numeric-only; RAW)
if 'class' in df.columns:
    ct = pd.concat([numeric_df, df[['class']]], axis=1)
    cols = [c for c in ct.columns if ct[c].notna().sum() >= 2]
    if 'class' in cols and len(cols) > 1:
        corr_target = ct[cols].corr(min_periods=2)['class'].sort_values(ascending=False)
        print("\nCorrelation with target 'class' (numeric-only, RAW):")
        print(corr_target)
    else:
        print("[INFO] Not enough usable columns to correlate with target.")
else:
    print("[INFO] No 'class' column for target correlation.")

# Class ratio (RAW)
ratio = df['class'].value_counts(normalize=True).mul(100).sort_index()
print("\nClass ratio (%) — RAW:\n", ratio)

plt.figure(figsize=(6, 4))
ax = ratio.plot(kind='bar', color=['orange','green'])
plt.title("Class Ratio (%) — RAW"); plt.ylabel("Percentage"); plt.xlabel("Class (0=not CKD, 1=CKD)")
plt.xticks(rotation=0)
for p in ax.patches:
    plt.annotate(f'{p.get_height():.1f}%',
                 (p.get_x()+p.get_width()/2., p.get_height()),
                 ha='center', va='bottom', fontsize=10, xytext=(0,3), textcoords='offset points')
plt.tight_layout(); plt.show()

# Class-wise numeric means (RAW)
class_means = df.groupby('class')[NUM_CONTINUOUS].mean()
print("\nMean of numeric features by class (RAW):")
print(class_means)

plt.figure(figsize=(14, 8))
sns.heatmap(class_means.T, annot=True, cmap='coolwarm', fmt=".2f", cbar_kws={'shrink':0.8})
plt.title("Numeric Feature Means by Class (RAW)")
plt.xlabel("Class (0=not CKD, 1=CKD)"); plt.ylabel("Feature"); plt.tight_layout(); plt.show()


# -------- 4) Split FIRST (prevents leakage) --------
df = df.dropna(subset=['class'])
X = df[NUM_CONTINUOUS + BIN_CATEGORICAL].copy()
y = df['class'].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
print(f"\nShapes -> X_train:{X_train.shape}  X_test:{X_test.shape}")
print(f"Targets -> y_train:{y_train.shape}  y_test:{y_test.shape}")


# -------- 5) Early Encoding (fit on train, transform both) --------
# 5a) Impute continuous numeric and binary categoricals separately (fit on train)
num_imputer = SimpleImputer(strategy='median')
bin_imputer = SimpleImputer(strategy='most_frequent')

Xtr_num = pd.DataFrame(num_imputer.fit_transform(X_train[NUM_CONTINUOUS]),
                       columns=NUM_CONTINUOUS, index=X_train.index)
Xte_num = pd.DataFrame(num_imputer.transform(X_test[NUM_CONTINUOUS]),
                       columns=NUM_CONTINUOUS, index=X_test.index)

Xtr_cat_raw = pd.DataFrame(bin_imputer.fit_transform(X_train[BIN_CATEGORICAL]),
                           columns=BIN_CATEGORICAL, index=X_train.index)
Xte_cat_raw = pd.DataFrame(bin_imputer.transform(X_test[BIN_CATEGORICAL]),
                           columns=BIN_CATEGORICAL, index=X_test.index)

# 5b) Map ALL binary categoricals to 0/1 (no *_yes columns, ever)
def map_binary(df_in, maps):
    df_out = df_in.copy()
    for c, m in maps.items():
        df_out[c] = df_out[c].map(m)
    # If any unexpected labels remain, coerce to NaN then fill with 0
    for c in df_out.columns:
        df_out[c] = pd.to_numeric(df_out[c], errors='coerce').fillna(0)
    return df_out.astype(int)

Xtr_bin = map_binary(Xtr_cat_raw, binary_maps)
Xte_bin = map_binary(Xte_cat_raw, binary_maps)

# 5c) Train-only z-cap + MinMax on continuous numeric ONLY (leave 0/1 as-is)
zcap = ZScoreWinsorizer(k=3.0)
Xtr_num_z = zcap.fit_transform(Xtr_num)
Xte_num_z = zcap.transform(Xte_num)

scaler = MinMaxScaler()
Xtr_num_s = pd.DataFrame(scaler.fit_transform(Xtr_num_z),
                         columns=NUM_CONTINUOUS, index=X_train.index)
Xte_num_s = pd.DataFrame(scaler.transform(Xte_num_z),
                         columns=NUM_CONTINUOUS, index=X_test.index)

# 5d) Final encoded matrices: scaled numeric + binary 0/1
X_train_enc = pd.concat([Xtr_num_s, Xtr_bin], axis=1)
X_test_enc  = pd.concat([Xte_num_s, Xte_bin], axis=1)

print("\nEncoded feature shapes -> X_train_enc:", X_train_enc.shape, " X_test_enc:", X_test_enc.shape)
print("Sample encoded columns:", list(X_train_enc.columns)[:15], "...")
print("\n[Preview] First 5 rows of scaled numeric (train):")
print(Xtr_num_s.head())


# -------- 6) Visualize class balance (train) + SMOTE (train only) --------
print("\nTraining class distribution (Before SMOTE):\n", y_train.value_counts())

plt.figure(figsize=(6,4))
sns.countplot(x=y_train, hue=y_train, palette="Set2", legend=False)
plt.title("Train Distribution — Before SMOTE"); plt.xlabel("Class"); plt.ylabel("Count")
plt.tight_layout(); plt.show()

sm = SMOTE(random_state=RANDOM_STATE)
Xtr_sm, ytr_sm = sm.fit_resample(X_train_enc, y_train)

print("\nTraining class distribution (After SMOTE):\n", pd.Series(ytr_sm).value_counts())

plt.figure(figsize=(6,4))
sns.countplot(x=ytr_sm, hue=ytr_sm, palette="Set2", legend=False)
plt.title("Train Distribution — After SMOTE"); plt.xlabel("Class"); plt.ylabel("Count")
plt.tight_layout(); plt.show()

print(f"\nShapes after SMOTE (TRAIN only): Xtr_sm={Xtr_sm.shape}, ytr_sm={ytr_sm.shape}")
print(f"Test remains unchanged: X_test_enc={X_test_enc.shape}, y_test={y_test.shape}")

# 3-panel: Before SMOTE, Overall RAW ratio, After SMOTE
plt.figure(figsize=(16, 4.8))
plt.suptitle("Class Distribution — Training & Overall", fontsize=14, y=0.98)

plt.subplot(1, 3, 1)
ax1 = sns.countplot(x=y_train, hue=y_train, palette='Set2', legend=False)
plt.title("Before SMOTE (Train)")
for p in ax1.patches:
    plt.annotate(f'{p.get_height()}', (p.get_x()+p.get_width()/2., p.get_height()),
                 ha='center', va='bottom', fontsize=9, xytext=(0,3), textcoords='offset points')

plt.subplot(1, 3, 2)
ratio.plot(kind='bar', color=['orange','green'])
plt.title("Overall Class Ratio (RAW)")
plt.ylabel("Percentage"); plt.xlabel("Class"); plt.xticks(rotation=0)
for i, v in enumerate(ratio.values):
    plt.text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=9)

plt.subplot(1, 3, 3)
ax3 = sns.countplot(x=ytr_sm, hue=ytr_sm, palette='Set2', legend=False)
plt.title("After SMOTE (Train)")
for p in ax3.patches:
    plt.annotate(f'{p.get_height()}', (p.get_x()+p.get_width()/2., p.get_height()),
                 ha='center', va='bottom', fontsize=9, xytext=(0,3), textcoords='offset points')

plt.tight_layout(rect=[0, 0, 1, 0.93]); plt.show()


# -------- 7) Random Forest Feature Importance on encoded train --------
rf = RandomForestClassifier(
    n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1, max_features='sqrt'
)
rf.fit(Xtr_sm, ytr_sm)

fi = pd.Series(rf.feature_importances_, index=Xtr_sm.columns).sort_values(ascending=False)

# Attractive gradient + colorbar (fixed: attach to the same Axes)
vals = fi.values
idxs = fi.index
norm = mcolors.Normalize(vmin=vals.min(), vmax=vals.max())
colors = cm.get_cmap('viridis')(norm(vals))

fig, ax = plt.subplots(figsize=(10, max(6, 0.35 * len(fi))))
bars = ax.barh(idxs, vals, color=colors, edgecolor='black', linewidth=0.6)
ax.set_title("Feature Importance")
ax.set_xlabel("Importance"); ax.set_ylabel("Feature")
ax.invert_yaxis()

for bar, v in zip(bars, vals):
    ax.annotate(f"{v:.3f}",
                xy=(v, bar.get_y() + bar.get_height()/2),
                xytext=(5, 0), textcoords='offset points',
                va='center', ha='left', fontsize=9)

sm = cm.ScalarMappable(cmap='viridis', norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=ax).set_label('Relative Importance', rotation=270, labelpad=12)

plt.tight_layout(); plt.show()

print("\nTop 24 features:\n", fi.head(24))

## Feature Selection (UCI CKD)

In [ ]:
# =========================================================
# Leakage-Safe Feature Selection (Train-only fitting)
# Uses: X_train_enc, X_test_enc, y_train, y_test  (from previous preprocessing step)
# Produces: common_features, X_train_common_selected, X_test_common_selected
# =========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE, SelectKBest, chi2, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# -------- Config --------
TOP_N_FEATURES    = 10          # number selected per method
MAJORITY_VOTE_K   = 3           # feature must appear in >= K methods
RANDOM_STATE      = 42

# -------- Safety checks --------
req = ['X_train_enc', 'X_test_enc', 'y_train']
for v in req:
    if v not in globals():
        raise RuntimeError(f"Missing required variable: {v}. Run preprocessing cell first.")

if list(X_train_enc.columns) != list(X_test_enc.columns):
    raise ValueError("Train/Test encoded feature names do not match.")

# Cap k to available features
k = min(TOP_N_FEATURES, X_train_enc.shape[1])

# -------- Train-only scaling snapshots --------
# For LR/RFE/MI/PCA → standardization (train-only)
std_scaler = StandardScaler()
Xtr_std = pd.DataFrame(std_scaler.fit_transform(X_train_enc),
                       columns=X_train_enc.columns, index=X_train_enc.index)
Xte_std = pd.DataFrame(std_scaler.transform(X_test_enc),
                       columns=X_test_enc.columns, index=X_test_enc.index)

# For Chi2 → strictly non-negative (train-only)
mm_scaler = MinMaxScaler()
Xtr_mm = pd.DataFrame(mm_scaler.fit_transform(X_train_enc),
                      columns=X_train_enc.columns, index=X_train_enc.index)
Xte_mm = pd.DataFrame(mm_scaler.transform(X_test_enc),
                      columns=X_test_enc.columns, index=X_test_enc.index)

# =================== Helper Selectors (fit on TRAIN only) ===================
def select_via_log_reg(X, y, top_n):
    lr = LogisticRegression(
        max_iter=10000, random_state=RANDOM_STATE,
        class_weight='balanced', solver='liblinear'
    )
    lr.fit(X, y)
    coef = np.abs(lr.coef_[0])
    top_idx = np.argsort(coef)[-top_n:]
    return X.columns[top_idx].to_numpy()

def select_via_rfe(X, y, top_n):
    lr = LogisticRegression(
        max_iter=10000, random_state=RANDOM_STATE,
        class_weight='balanced', solver='liblinear'
    )
    rfe = RFE(estimator=lr, n_features_to_select=top_n)
    rfe.fit(X, y)
    return X.columns[rfe.support_].to_numpy()

def select_via_random_forest(X, y, top_n):
    rf = RandomForestClassifier(
        n_estimators=300, random_state=RANDOM_STATE,
        n_jobs=-1, class_weight='balanced_subsample', max_features='sqrt'
    )
    rf.fit(X, y)
    imp = rf.feature_importances_
    top_idx = np.argsort(imp)[-top_n:]
    return X.columns[top_idx].to_numpy()

def select_via_mutual_info(X, y, top_n):
    mi = mutual_info_classif(X, y, random_state=RANDOM_STATE)
    top_idx = np.argsort(mi)[-top_n:]
    return X.columns[top_idx].to_numpy()

def select_via_chi2(X_nonneg, y, top_n):
    selector = SelectKBest(score_func=chi2, k=top_n)
    selector.fit(X_nonneg, y)
    return X_nonneg.columns[selector.get_support(indices=True)].to_numpy()

def select_via_pca(X, top_n):
    pca = PCA(n_components=min(top_n, X.shape[1]), random_state=RANDOM_STATE)
    pca.fit(X)
    # Aggregate absolute loadings across PCs to score features
    importance = np.abs(pca.components_).sum(axis=0)
    top_idx = np.argsort(importance)[-top_n:]
    return X.columns[top_idx].to_numpy()

# =================== Run selectors (TRAIN ONLY) ===================
log_reg_selected = select_via_log_reg(Xtr_std, y_train, k)
rfe_selected     = select_via_rfe(Xtr_std, y_train, k)
rf_selected      = select_via_random_forest(X_train_enc, y_train, k)  # trees: no scaling needed
mi_selected      = select_via_mutual_info(Xtr_std, y_train, k)
chi2_selected    = select_via_chi2(Xtr_mm, y_train, k)                # chi2 requires non-negative
pca_selected     = select_via_pca(Xtr_std, k)

# =================== Display Selected Features ===================
print("\nFeatures Selected via Logistic Regression:")
print(pd.Index(log_reg_selected))
print("\nFeatures Selected via RFE:")
print(pd.Index(rfe_selected))
print("\nFeatures Selected via Random Forest:")
print(pd.Index(rf_selected))
print("\nFeatures Selected via Mutual Information:")
print(pd.Index(mi_selected))
print("\nFeatures Selected via Chi-Square:")
print(pd.Index(chi2_selected))
print("\nFeatures Selected via PCA:")
print(pd.Index(pca_selected))

# =================== Majority Vote (appears in ≥ MAJORITY_VOTE_K) ===================
all_feats = np.concatenate([log_reg_selected, rfe_selected, rf_selected,
                            mi_selected, chi2_selected, pca_selected])
counts = Counter(all_feats)
common_features = [f for f, c in counts.items() if c >= MAJORITY_VOTE_K]
# Keep original column order for readability
common_features = [c for c in X_train_enc.columns if c in common_features]

print(f"\nCommon Features (appearing in ≥{MAJORITY_VOTE_K} methods):")
print(pd.Index(common_features))

# Optional: visualize vote counts
vote_df = pd.DataFrame({'feature': list(counts.keys()),
                        'votes':   list(counts.values())}).sort_values('votes', ascending=False)

plt.figure(figsize=(10, max(4, 0.35*len(vote_df))))
ax = sns.barplot(y='feature', x='votes', hue='feature', data=vote_df, palette='viridis', orient='h', legend=False)
plt.title("Feature Vote Counts Across Selection Methods")
plt.xlabel("Votes"); plt.ylabel("Feature")
for p in ax.patches:
    v = p.get_width()
    ax.annotate(f"{int(v)}", (v, p.get_y()+p.get_height()/2),
                ha='left', va='center', xytext=(4,0), textcoords='offset points', fontsize=9)
plt.tight_layout(); plt.show()

# =================== Create Train/Test with Common Features ===================
# (No SMOTE here. If you upsample, do it later on TRAIN only.)
X_train_common_selected = X_train_enc[common_features].copy()
X_test_common_selected  = X_test_enc[common_features].copy()

print(f"\nShape — Training (common features): {X_train_common_selected.shape}")
print(f"Shape — Test (common features):     {X_test_common_selected.shape}")

## Modeling (UCI CKD)

In [ ]:
import os
# =========================================================
# CKD Modeling — Tuned, Leakage-Safe (No AUC/ROC)
# - Train/Test table: Accuracy, Precision, Recall, F1_Score, MCR
# - Comprehensive Algorithm Comparison — Train vs Test (SEPARATE FIGURES)
# - Bootstrap 95% CI plots — SEPARATE FIGURES
# - PRINTED Bootstrap CI table + Combined Summary (no overlap)
# - Confusion matrices + classification reports
# NOTE: SMOTE is applied ONLY inside training via pipelines (leakage-safe).
# =========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

# Optional XGBoost
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

# -------- Styling --------
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

METRIC_ORDER = ["Accuracy", "Precision", "Recall", "F1_Score", "MCR"]

# Colors
COLORS_TRAIN_TEST = ["#2F80ED", "#F2994A"]  # Train (blue) vs Test (orange)
METRIC_COLORS = {
    "Accuracy": "#2F80ED",
    "Precision": "#F2994A",
    "Recall": "#27AE60",
    "F1_Score": "#9B51E0",
    "MCR": "#EB5757",
}
CMAP_CONFMAT = "Blues"

RANDOM_STATE = 42
N_BOOT = int(os.getenv("CKD_BOOTSTRAP_REPS", "2000"))  # original default: 2000; reduced reps for smoke tests only

# -------- Feature matrices from previous step --------
if 'X_train_common_selected' in globals() and 'X_test_common_selected' in globals():
    Xtr = X_train_common_selected.copy()
    Xte = X_test_common_selected.copy()
else:
    Xtr = X_train_enc.copy()
    Xte = X_test_enc.copy()

ytr = y_train.copy()
yte = y_test.copy()

# -------- Utilities --------
def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1v = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    mcr = 1.0 - acc
    return acc, prec, rec, f1v, mcr

def bootstrap_ci(y_true, y_pred, n_boot=2000, alpha=0.05, random_state=42):
    rng = np.random.default_rng(random_state)
    n = len(y_true)
    idx = np.arange(n)
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    accs, precs, recs, f1s, mcrs = [], [], [], [], []
    for _ in range(n_boot):
        b = rng.choice(idx, size=n, replace=True)
        yt = y_true[b]; yp = y_pred[b]
        a, p, r, f, m = compute_metrics(yt, yp)
        accs.append(a); precs.append(p); recs.append(r); f1s.append(f); mcrs.append(m)
    def ci(arr):
        lo, hi = np.quantile(arr, [alpha/2, 1-alpha/2])
        return float(lo), float(hi)
    return {
        'accuracy':  ci(accs),
        'precision': ci(precs),
        'recall':    ci(recs),
        'f1':        ci(f1s),
        'mcr':       ci(mcrs),
    }

def label_bars(ax, fmt="{:.3f}", dy=6, fontsize=10):
    for p in ax.patches:
        try:
            v = p.get_height()
            x = p.get_x() + p.get_width()/2
            ax.annotate(fmt.format(v), (x, v),
                        ha='center', va='bottom',
                        fontsize=fontsize, xytext=(0, max(dy, 0.01)),
                        textcoords='offset points')
        except Exception:
            pass

# -------- Final tuned pipelines (SMOTE inside fit only) --------
models = OrderedDict()

models["KNN"] = ImbPipeline(steps=[
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('clf', KNeighborsClassifier(n_neighbors=3, metric='euclidean', weights='uniform'))
])

models["SVM (RBF)"] = ImbPipeline(steps=[
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('clf', SVC(kernel='rbf', probability=False, random_state=RANDOM_STATE,
                C=4.6415888336127775, gamma=0.1668100537200059))
])

models["Random Forest"] = ImbPipeline(steps=[
    ('scaler', 'passthrough'),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('clf', RandomForestClassifier(
        n_estimators=500, max_depth=8, max_features='log2',
        min_samples_split=2, min_samples_leaf=1,
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE, n_jobs=-1
    ))
])

models["Gaussian NB"] = ImbPipeline(steps=[
    ('scaler', 'passthrough'),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('clf', GaussianNB(var_smoothing=1e-12))
])

models["Logistic Regression"] = ImbPipeline(steps=[
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('clf', LogisticRegression(max_iter=10000, solver='liblinear',
                               penalty='l1', C=10.0,
                               class_weight='balanced',
                               random_state=RANDOM_STATE))
])

models["MLP"] = ImbPipeline(steps=[
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('clf', MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu',
                          alpha=0.03162277660168379,
                          learning_rate_init=0.00046415888336127773,
                          max_iter=800, random_state=RANDOM_STATE))
])

if HAS_XGB:
    models["XGBoost"] = ImbPipeline(steps=[
        ('scaler', 'passthrough'),
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('clf', XGBClassifier(
            n_estimators=600, max_depth=3, learning_rate=0.01,
            subsample=1.0, colsample_bytree=0.7, reg_lambda=1.0,
            objective='binary:logistic', eval_metric='logloss',
            random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])

# -------- Fit, evaluate, and collect results --------
rows = []
test_preds = {}

for name, pipe in models.items():
    print(f"\n=== Final Fit & Evaluate: {name} ===")
    pipe.fit(Xtr, ytr)
    ytr_pred = pipe.predict(Xtr)
    yte_pred = pipe.predict(Xte)
    tr_acc, tr_prec, tr_rec, tr_f1, tr_mcr = compute_metrics(ytr, ytr_pred)
    te_acc, te_prec, te_rec, te_f1, te_mcr = compute_metrics(yte, yte_pred)
    rows.append({
        'Model': name,
        'Train_Accuracy': tr_acc, 'Train_Precision': tr_prec, 'Train_Recall': tr_rec, 'Train_F1_Score': tr_f1, 'Train_MCR': tr_mcr,
        'Test_Accuracy':  te_acc, 'Test_Precision':  te_prec, 'Test_Recall':  te_rec, 'Test_F1_Score':  te_f1, 'Test_MCR':  te_mcr
    })
    test_preds[name] = (yte, yte_pred)

results_df = pd.DataFrame(rows).set_index('Model').round(3)
print("\n=== Train vs Test (Tuned Models; No AUC) ===")
print(results_df)

# =========================================================
# Comprehensive Algorithm Comparison — Train vs Test
# (SEPARATE FIGURES, one per metric)
# =========================================================
for metric in METRIC_ORDER:
    fig, ax = plt.subplots(figsize=(12, 6))
    train_col = f"Train_{metric}"
    test_col  = f"Test_{metric}"
    sub = results_df[[train_col, test_col]].copy()
    sub.columns = ["Train", "Test"]

    sub.plot(kind='bar', ax=ax, color=COLORS_TRAIN_TEST, edgecolor='white', width=0.75)
    ax.set_title(f"Comprehensive Algorithm Comparison — {metric}", pad=8, fontsize=15)
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xlabel("Model", fontsize=12)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.legend(loc='best', frameon=False)
    ax.set_ylim(0, 0.25 if metric == "MCR" else 1.05)
    label_bars(ax, fmt="{:.3f}", dy=6, fontsize=10)
    plt.xticks(rotation=20, ha='right', fontsize=10)
    plt.tight_layout()
    plt.show()

# =========================================================
# Bootstrap 95% CIs — SEPARATE FIGURES, one per metric
# =========================================================
ci_rows = []
for name, (yt, yp) in test_preds.items():
    cis = bootstrap_ci(yt, yp, n_boot=N_BOOT, alpha=0.05, random_state=RANDOM_STATE)
    ci_rows.append({
        'Model': name,
        'Accuracy': results_df.loc[name, 'Test_Accuracy'],
        'Accuracy_CI_low':  cis['accuracy'][0],
        'Accuracy_CI_high': cis['accuracy'][1],
        'Precision': results_df.loc[name, 'Test_Precision'],
        'Precision_CI_low':  cis['precision'][0],
        'Precision_CI_high': cis['precision'][1],
        'Recall': results_df.loc[name, 'Test_Recall'],
        'Recall_CI_low':  cis['recall'][0],
        'Recall_CI_high': cis['recall'][1],
        'F1_Score': results_df.loc[name, 'Test_F1_Score'],
        'F1_Score_CI_low':  cis['f1'][0],
        'F1_Score_CI_high': cis['f1'][1],
        'MCR': results_df.loc[name, 'Test_MCR'],
        'MCR_CI_low':  cis['mcr'][0],
        'MCR_CI_high': cis['mcr'][1],
    })
ci_df = pd.DataFrame(ci_rows).set_index('Model')

for metric in METRIC_ORDER:
    fig, ax = plt.subplots(figsize=(12, 6))
    vals  = ci_df[metric].values
    lows  = ci_df[f"{metric}_CI_low"].values
    highs = ci_df[f"{metric}_CI_high"].values
    yerr  = np.vstack([vals - lows, highs - vals])

    bars = ax.bar(ci_df.index, vals, color=METRIC_COLORS[metric], edgecolor='white', width=0.70)
    ax.errorbar(ci_df.index, vals, yerr=yerr, fmt='none',
                ecolor='black', capsize=5, linewidth=1.2)
    ax.set_ylim(0, 0.25 if metric == "MCR" else 1.05)
    ax.set_title(f"{metric} — 95% CI (Bootstrap, Test)", pad=8, fontsize=15)
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xlabel("Model", fontsize=12)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    plt.xticks(rotation=20, ha='right', fontsize=10)
    label_bars(ax, fmt="{:.3f}", dy=6, fontsize=10)
    plt.tight_layout()
    plt.show()

# =========================================================
# PRINT: Bootstrap CI Results (sorted, rounded)
# =========================================================
cols_order = [
    "Accuracy", "Accuracy_CI_low", "Accuracy_CI_high",
    "Precision", "Precision_CI_low", "Precision_CI_high",
    "Recall", "Recall_CI_low", "Recall_CI_high",
    "F1_Score", "F1_Score_CI_low", "F1_Score_CI_high",
    "MCR", "MCR_CI_low", "MCR_CI_high",
]
cols_order = [c for c in cols_order if c in ci_df.columns]

ci_print = ci_df.loc[:, cols_order].copy().round(3).sort_values("Accuracy", ascending=False)
print("\n=== Test Metrics with 95% Bootstrap CIs (Tuned; No AUC) ===")
print(ci_print)

# =========================================================
# Combined Summary (Test point estimates + 95% CIs) — no overlap
# =========================================================
print("\n=== Combined Summary (Test point estimates + 95% CIs) ===")
test_point = (results_df.filter(like="Test_")
              .rename(columns=lambda c: c.replace("Test_", ""))
              .add_suffix("_Point"))
combined = test_point.join(ci_df.loc[:, cols_order])
print(combined.round(3))

# -------- Confusion Matrices (Test) + Reports --------
for name, (yt, yp) in test_preds.items():
    cm = confusion_matrix(yt, yp)
    print(f"\n{name} — Confusion Matrix (Test):\n{cm}")
    sns.heatmap(cm, annot=True, fmt='d', cmap=CMAP_CONFMAT, cbar=False)
    plt.title(f"{name} — Confusion Matrix (Test)", pad=8)
    plt.xlabel("Predicted Label"); plt.ylabel("True Label")
    plt.tight_layout(); plt.show()

for name, (yt, yp) in test_preds.items():
    print(f"\n{name} — Classification Report (Test):")
    print(classification_report(yt, yp, digits=3))

## Stacking

In [ ]:
import os
# =========================================================
# Stacking Ensemble (Leakage-safe) + Ranking + SEPARATE Plots
# Custom plot order: Stacking first, then XGBoost
# =========================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

# Optional XGBoost
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

# -------- Style --------
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

RANDOM_STATE = 42
N_BOOT = int(os.getenv("CKD_BOOTSTRAP_REPS", "2000"))  # original default: 2000
METRIC_LIST = ["Accuracy", "Precision", "Recall", "F1_Score", "MCR"]

# Colors
COLORS_TRAIN_TEST = ["#1f77b4", "#ff7f0e"]  # Train, Test
CMAP_CONFMAT = "Blues"
METRIC_COLORS = {
    "Accuracy":  "#2ca02c",
    "Precision": "#9467bd",
    "Recall":    "#8c564b",
    "F1_Score":  "#d62728",
    "MCR":       "#17becf",
}

# ---------------------------------------------------------
# Safety check for required inputs
# ---------------------------------------------------------
if not all(k in globals() for k in ["Xtr", "Xte", "ytr", "yte"]):
    raise RuntimeError("Expected Xtr, Xte, ytr, yte to be defined from the previous step.")

# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------
def compute_metrics(y_true, y_pred):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1v  = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    mcr  = 1.0 - acc
    return acc, prec, rec, f1v, mcr

def bootstrap_ci(y_true, y_pred, n_boot=2000, alpha=0.05, random_state=42):
    rng = np.random.RandomState(random_state)
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    n = len(y_true)

    accs, precs, recs, f1s, mcrs = [], [], [], [], []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yt = y_true[idx]; yp = y_pred[idx]
        a, p, r, f, m = compute_metrics(yt, yp)
        accs.append(a); precs.append(p); recs.append(r); f1s.append(f); mcrs.append(m)

    def ci(vals):
        lo = np.percentile(vals, 100*alpha/2.0)
        hi = np.percentile(vals, 100*(1 - alpha/2.0))
        return lo, hi

    return {"accuracy": ci(accs), "precision": ci(precs),
            "recall": ci(recs), "f1": ci(f1s), "mcr": ci(mcrs)}

def visualize_metrics_separate(results_df, metrics, colors):
    """One HBAR figure per metric, with value labels."""
    for metric, color in zip(metrics, colors):
        fig, ax = plt.subplots(figsize=(12, 6))
        bars = ax.barh(results_df.index, results_df[metric], color=color, alpha=0.9, edgecolor="white")
        ax.set_xlabel(metric, fontsize=12)
        ax.set_title(f"Model Comparison — {metric}", fontsize=15, pad=6)
        ax.grid(axis='x', linestyle='--', alpha=0.4)
        for b in bars:
            w = b.get_width()
            ax.text(w + 0.01, b.get_y() + b.get_height()/2, f"{w:.3f}",
                    va="center", ha="left", fontsize=10)
        plt.tight_layout()
        plt.show()

def rank_models(results_df, use="Test"):
    """
    Rank by Test_Accuracy (desc), Test_Precision (desc), Test_Recall (desc),
    Test_F1_Score (desc), Test_MCR (asc).
    """
    df = results_df.copy()
    cols = [f"{use}_Accuracy", f"{use}_Precision", f"{use}_Recall", f"{use}_F1_Score", f"{use}_MCR"]
    missing = [c for c in cols if c not in df.columns]
    if len(missing) > 0:
        raise ValueError(f"Missing columns for ranking: {missing}")

    df["Accuracy Rank"]   = df[f"{use}_Accuracy"].rank(ascending=False, method="min")
    df["Precision Rank"]  = df[f"{use}_Precision"].rank(ascending=False, method="min")
    df["Recall Rank"]     = df[f"{use}_Recall"].rank(ascending=False, method="min")
    df["F1_Score Rank"]   = df[f"{use}_F1_Score"].rank(ascending=False, method="min")
    df["MCR Rank"]        = df[f"{use}_MCR"].rank(ascending=True,  method="min")
    df["Total Rank"]      = df[["Accuracy Rank","Precision Rank","Recall Rank","F1_Score Rank","MCR Rank"]].sum(axis=1)
    df = df.sort_values("Total Rank", ascending=True)
    return df

def reorder_for_plots(index_like, first="Stacking Ensemble", second="XGBoost"):
    """
    Put `first` first, `second` second (if present), then the rest in given order.
    """
    idx = list(index_like)
    out = []
    if first in idx:
        out.append(first); idx.remove(first)
    if second in idx:
        out.append(second); idx.remove(second)
    out.extend(idx)
    return out

def label_bars(ax, fmt="{:.3f}", dy=6, fontsize=10):
    for p in ax.patches:
        try:
            v = p.get_height()
            x = p.get_x() + p.get_width()/2
            ax.annotate(fmt.format(v), (x, v),
                        ha='center', va='bottom',
                        fontsize=fontsize, xytext=(0, max(dy, 0.01)),
                        textcoords='offset points')
        except Exception:
            pass

# ---------------------------------------------------------
# Build tuned base learners (your tuned params)
# ---------------------------------------------------------
base_estimators = [
    ('knn', KNeighborsClassifier(n_neighbors=3, metric='euclidean', weights='uniform')),
    ('svm', SVC(kernel='rbf', probability=True,
                C=4.6415888336127775, gamma=0.1668100537200059, random_state=RANDOM_STATE)),
    ('rf',  RandomForestClassifier(
        n_estimators=500, max_depth=8, max_features='log2',
        min_samples_split=2, min_samples_leaf=1,
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE, n_jobs=-1
    )),
    ('gnb', GaussianNB(var_smoothing=1e-12)),
    ('lr',  LogisticRegression(max_iter=10000, solver='liblinear',
                               penalty='l1', C=10.0,
                               class_weight='balanced',
                               random_state=RANDOM_STATE)),
    ('mlp', MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu',
                          alpha=0.03162277660168379,
                          learning_rate_init=0.00046415888336127773,
                          max_iter=800, random_state=RANDOM_STATE))
]
if HAS_XGB:
    base_estimators.append(
        ('xgb', XGBClassifier(
            n_estimators=600, max_depth=3, learning_rate=0.01,
            subsample=1.0, colsample_bytree=0.7, reg_lambda=1.0,
            objective='binary:logistic', eval_metric='logloss',
            random_state=RANDOM_STATE, n_jobs=-1
        ))
    )

meta_lr = LogisticRegression(
    max_iter=10000, solver='liblinear', penalty='l2',
    class_weight='balanced', random_state=RANDOM_STATE
)

stack_clf = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta_lr,
    stack_method='predict_proba',
    passthrough=False,
    cv=5,
    n_jobs=-1
)

# Leakage-safe: scale -> SMOTE (train only) -> stack
stacking_pipe = ImbPipeline(steps=[
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('stack', stack_clf)
])

# ---------------------------------------------------------
# Fit & Evaluate Stacking
# ---------------------------------------------------------
print("\n=== Final Fit & Evaluate: Stacking Ensemble ===")
stacking_pipe.fit(Xtr, ytr)  # SMOTE applies inside fit on training folds

ytr_pred = stacking_pipe.predict(Xtr)
yte_pred = stacking_pipe.predict(Xte)

tr_acc, tr_prec, tr_rec, tr_f1, tr_mcr = compute_metrics(ytr, ytr_pred)
te_acc, te_prec, te_rec, te_f1, te_mcr = compute_metrics(yte, yte_pred)

print("\n[Stacking Ensemble] Train metrics:")
print(f"Accuracy={tr_acc:.3f}  Precision={tr_prec:.3f}  Recall={tr_rec:.3f}  F1_Score={tr_f1:.3f}  MCR={tr_mcr:.3f}")

print("\n[Stacking Ensemble] Test metrics:")
print(f"Accuracy={te_acc:.3f}  Precision={te_prec:.3f}  Recall={te_rec:.3f}  F1_Score={te_f1:.3f}  MCR={te_mcr:.3f}")

cis = bootstrap_ci(yte, yte_pred, n_boot=N_BOOT, alpha=0.05, random_state=RANDOM_STATE)
stack_ci_row = pd.DataFrame({
    'Model': ["Stacking Ensemble"],
    'Accuracy': [te_acc], 'Accuracy_CI_low': [cis['accuracy'][0]], 'Accuracy_CI_high': [cis['accuracy'][1]],
    'Precision': [te_prec], 'Precision_CI_low': [cis['precision'][0]], 'Precision_CI_high': [cis['precision'][1]],
    'Recall': [te_rec], 'Recall_CI_low': [cis['recall'][0]], 'Recall_CI_high': [cis['recall'][1]],
    'F1_Score': [te_f1], 'F1_Score_CI_low': [cis['f1'][0]], 'F1_Score_CI_high': [cis['f1'][1]],
    'MCR': [te_mcr], 'MCR_CI_low': [cis['mcr'][0]], 'MCR_CI_high': [cis['mcr'][1]],
}).set_index('Model')

# ---------------------------------------------------------
# Update / Create results tables
# (Assume results_df & ci_df from earlier models exist; else create)
# ---------------------------------------------------------
stack_row = pd.DataFrame({
    'Model': ["Stacking Ensemble"],
    'Train_Accuracy': [tr_acc], 'Train_Precision': [tr_prec], 'Train_Recall': [tr_rec], 'Train_F1_Score': [tr_f1], 'Train_MCR': [tr_mcr],
    'Test_Accuracy':  [te_acc], 'Test_Precision':  [te_prec], 'Test_Recall':  [te_rec], 'Test_F1_Score':  [te_f1], 'Test_MCR':  [te_mcr]
}).set_index('Model')

if 'results_df' in globals():
    results_df = pd.concat([results_df, stack_row], axis=0)
else:
    results_df = stack_row.copy()

if 'ci_df' in globals():
    ci_df = pd.concat([ci_df, stack_ci_row], axis=0)
else:
    ci_df = stack_ci_row.copy()

# De-duplicate indices (keep last) to avoid reindex clashes
results_df = results_df[~results_df.index.duplicated(keep='last')].copy()
ci_df      = ci_df[~ci_df.index.duplicated(keep='last')].copy()

print("\n=== Updated Train vs Test (with Stacking) ===")
print(results_df.round(3))

# ---------------------------------------------------------
# Ranking + Identify Best Model
# ---------------------------------------------------------
rank_basis = results_df[['Test_Accuracy','Test_Precision','Test_Recall','Test_F1_Score','Test_MCR']].copy()
rank_basis.columns = ['Accuracy','Precision','Recall','F1_Score','MCR']

ranked_df = rank_models(results_df, use="Test")
best_row = ranked_df.iloc[0]
best_model_name = best_row.name
print(f"\nBest Overall Model (by Test metrics Total Rank): {best_model_name}  |  Total Rank = {best_row['Total Rank']:.0f}")

print("\nRanked Model Comparison (lower Total Rank = better):")
print(ranked_df[['Accuracy Rank','Precision Rank','Recall Rank','F1_Score Rank','MCR Rank','Total Rank']].round(0))

# ---------------------------------------------------------
# Custom plot order: Stacking first, then XGBoost, then others by rank
# ---------------------------------------------------------
plot_order = reorder_for_plots(ranked_df.index, first="Stacking Ensemble", second="XGBoost")

# ---------------------------------------------------------
# Ranking plots (SEPARATE figures), requested colors
# ---------------------------------------------------------
visualize_metrics_separate(
    rank_basis.reindex(plot_order),
    ["Accuracy", "Precision", "Recall", "F1_Score", "MCR"],
    ["blue", "orange", "green", "red", "purple"]
)

# ---------------------------------------------------------
# Comprehensive Algorithm Comparison — Train vs Test
# (SEPARATE FIGURES, one per metric)
# ---------------------------------------------------------
for metric in METRIC_LIST:
    fig, ax = plt.subplots(figsize=(12, 6))
    train_col = f"Train_{metric}"
    test_col  = f"Test_{metric}"

    sub = results_df.loc[plot_order, [train_col, test_col]].copy()
    sub.columns = ["Train", "Test"]

    sub.plot(kind='bar', ax=ax, color=COLORS_TRAIN_TEST, edgecolor='white', width=0.78)
    ax.set_title(f"Comprehensive Algorithm Comparison — {metric}", pad=8, fontsize=15)
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xlabel("Model", fontsize=12)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.legend(loc='best', frameon=False)
    ax.set_ylim(0, 0.25 if metric == "MCR" else 1.05)

    for p in ax.patches:
        h = p.get_height()
        x = p.get_x() + p.get_width()/2
        ax.text(x, h + (0.01 if metric != "MCR" else 0.002), f"{h:.3f}",
                ha='center', va='bottom', fontsize=10)

    plt.xticks(rotation=20, ha='right', fontsize=10)
    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------
# Bootstrap 95% CIs — SEPARATE FIGURES per metric
# ---------------------------------------------------------
cols_order = [
    "Accuracy", "Accuracy_CI_low", "Accuracy_CI_high",
    "Precision", "Precision_CI_low", "Precision_CI_high",
    "Recall", "Recall_CI_low", "Recall_CI_high",
    "F1_Score", "F1_Score_CI_low", "F1_Score_CI_high",
    "MCR", "MCR_CI_low", "MCR_CI_high",
]

ci_df_sorted = ci_df.loc[plot_order]  # same custom order

for metric in METRIC_LIST:
    fig, ax = plt.subplots(figsize=(12, 6))
    vals  = ci_df_sorted[metric].values
    lows  = ci_df_sorted[f"{metric}_CI_low"].values
    highs = ci_df_sorted[f"{metric}_CI_high"].values
    yerr  = np.vstack([vals - lows, highs - vals])

    bars = ax.bar(ci_df_sorted.index, vals, color=METRIC_COLORS[metric], edgecolor='white', width=0.70)
    ax.errorbar(ci_df_sorted.index, vals, yerr=yerr, fmt='none',
                ecolor='black', capsize=5, linewidth=1.2)
    ax.set_ylim(0, 0.25 if metric == "MCR" else 1.05)
    ax.set_title(f"{metric} — 95% CI (Bootstrap, Test)", pad=8, fontsize=15)
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xlabel("Model", fontsize=12)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

    for p in bars:
        h = p.get_height()
        ax.text(p.get_x() + p.get_width()/2, h + (0.01 if metric != "MCR" else 0.002),
                f"{h:.3f}", ha='center', va='bottom', fontsize=10)

    plt.xticks(rotation=20, ha='right', fontsize=10)
    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------
# Print CI rows
# ---------------------------------------------------------
print("\n[Stacking Ensemble] 95% Bootstrap CIs:")
print(stack_ci_row.loc[["Stacking Ensemble"], cols_order].round(3))

print("\n=== Test Metrics with 95% Bootstrap CIs (All Models incl. Stacking) ===")
print(ci_df.loc[:, cols_order].copy().round(3).sort_values("Accuracy", ascending=False))

# ---------------------------------------------------------
# Confusion Matrix + Classification Report for Stacking
# ---------------------------------------------------------
cm = confusion_matrix(yte, yte_pred)
print("\nStacking Ensemble — Confusion Matrix (Test):\n", cm)
sns.heatmap(cm, annot=True, fmt='d', cmap=CMAP_CONFMAT, cbar=False)
plt.title("Stacking Ensemble — Confusion Matrix (Test)", pad=8)
plt.xlabel("Predicted Label"); plt.ylabel("True Label")
plt.tight_layout(); plt.show()

print("\nStacking Ensemble — Classification Report (Test):")
print(classification_report(yte, yte_pred, digits=3))